In [1]:
import argparse
import os
import sys

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import torch
torch.multiprocessing.set_start_method('spawn')

import jax
from lob.encoding import Vocab, Message_Tokenizer

from lob import inference_no_errcorr as inference
from lob.init_train import init_train_state, load_checkpoint, load_metadata, load_args_from_checkpoint

from lob import inference_no_errcorr as inference
import lob.encoding as encoding
import preproc as preproc

import jax.numpy as jnp
import numpy as onp

from pathlib import Path

import pandas as pd

from datetime import datetime
import yaml

import historical_scenario
import numpy as np
from tqdm import tqdm

from flax.training.train_state import TrainState
from lob.lobster_dataloader import LOBSTER_Dataset
import flax.linen as nn
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple, Union
import json

CUDA backend failed to initialize: jaxlib/cuda/versions_helpers.cc:98: operation cuInit(0) failed: CUDA_ERROR_NO_DEVICE (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)
/opt/conda/envs/myenv/lib/python3.12/site-packages/torch/cuda/__init__.py:619: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
2025-10-06 22:13:21.864831: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
def parse_args(config_file):

    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--config",
        type=str,
        default=config_file,
        help="Path to your YAML config file"
    )

    # Используем parse_known_args вместо parse_args
    args, _ = parser.parse_known_args()
    return args

In [3]:
args = parse_args("1_run_exp_aggresive_scenario.yaml")
with open(args.config, "r") as f:
    cfg = yaml.safe_load(f)

In [4]:
save_folder       = cfg["save_folder"]
batch_size        = cfg["batch_size"]
n_samples         = cfg["n_samples"]
n_gen_msgs        = cfg["n_gen_msgs"]
midprice_step_size= cfg["midprice_step_size"]
num_insertions    = cfg["num_insertions"]
num_coolings      = cfg["num_coolings"]
EVENT_TYPE_i      = cfg["EVENT_TYPE_i"]
DIRECTION_i       = cfg["DIRECTION_i"]
order_volume      = cfg["order_volume"]
bsz               = cfg["bsz"]
n_messages        = cfg["n_messages"]
book_dim          = cfg["book_dim"]
n_vol_series      = cfg["n_vol_series"]
sample_top_n      = cfg["sample_top_n"]
model_size        = cfg["model_size"]
data_dir          = cfg["data_dir"]
sample_all        = cfg["sample_all"]
stock             = cfg["stock"]
tick_size         = cfg["tick_size"]
rng_seed          = cfg["rng_seed"]
ckpt_path         = cfg["ckpt_path"]
num_devices = jax.local_device_count()
print(f'num_devices: ', num_devices)

use_sample_file  = cfg["use_sample_file"]
sample_file_path = cfg["sample_file_path"]
start_batch      = cfg["start_batch"]
end_batch        = cfg["end_batch"]

order_volume = cfg["order_volume"]
order_volume_ratio = cfg["order_volume_ratio"]
use_relative_volume = cfg["use_relative_volume"]

start_batch = 0
end_batch = 64

# batch_size = 4
# n_samples = 8
# bsz = 25
# n_gen_msgs = 50
# midprice_step_size = 50
# num_insertions = 2
# num_coolings = 2
# EVENT_TYPE_i = 4
# DIRECTION_i = 0
# order_volume = 75

num_devices:  1


In [5]:
print(num_insertions)
print(num_coolings)
print(use_sample_file)
print(sample_file_path)
print(start_batch)
print(end_batch)

100
20
True
bathces_equal_sample_indices_b64_bs16_ins10_cool50.json
0
64


In [6]:
# Load metadata and model
print("Loading metadata from", ckpt_path)
args_ckpt = load_metadata(ckpt_path)
print("Initializing model...")
train_state, model_cls = init_train_state(
    args_ckpt,
    n_classes=len(Vocab()),
    seq_len=n_messages * Message_Tokenizer.MSG_LEN,
    book_dim=book_dim,
    book_seq_len=n_messages,
)

import jax
from lob import init_train

def safe_deduplicate_trainstate(state):
    try:
        devices = jax.devices("gpu")
    except RuntimeError:
        devices = jax.devices("cpu")
        print("[INFO] GPU not available. Falling back to CPU.")
    else:
        print("[INFO] Running on GPU.")
    
    return jax.device_put(
        jax.tree.map(lambda x: x[0], state),
        device=devices[0]
    )

init_train.deduplicate_trainstate = safe_deduplicate_trainstate
print("Loading checkpoint...")
ckpt = load_checkpoint(train_state, ckpt_path, train=False)
state = ckpt["model"]
model = model_cls(training=False, step_rescale=1.0)

Loading metadata from checkpoints/denim-elevator-754_czg1ss71/
Initializing model...
configuring standard optimization setup
[*] Trainable Parameters: 35776312
Loading checkpoint...
[INFO] GPU not available. Falling back to CPU.


In [7]:
# prepare RNG
rng = jax.random.PRNGKey(rng_seed)

# data directory
data_path = Path(data_dir) / stock
data_path.mkdir(parents=True, exist_ok=True)
print(f"Data directory: {data_path} ({len(list(data_path.iterdir()))} files)")

# Experiment upload folder
exp_folder = historical_scenario.create_next_experiment_folder(save_folder)
print("Experiment dir:", exp_folder)
with open(exp_folder / "used_config.yaml", "w") as f_out:
    yaml.dump(cfg, f_out)

Data directory: /app/data/test_set/GOOG (37 files)
Experiment dir: /app/data_saved/exp_154_20251006_221422


In [8]:
# get dataset
ds = inference.get_dataset(data_path, n_messages, (num_insertions + num_coolings) * n_gen_msgs)

In [9]:
len(ds)

3337

In [10]:
sample_idx = 3311
date = ds.get_date(sample_idx)
print(f"Sample {sample_idx} is from date: {date}")

Sample 3311 is from date: 2023-01-13


# ================= Experiment indexes =================

In [11]:
# Create a DataFrame to analyze date distribution
import pandas as pd

# Get all sample indices and their corresponding dates
all_indices = list(range(len(ds)))
date_data = []
for idx in all_indices:
    date = ds.get_date(idx)
    date_data.append({'sample_idx': idx, 'date': date})

df = pd.DataFrame(date_data)
print("Date distribution in dataset:")
date_counts = df['date'].value_counts().sort_index()
print(date_counts)

# Calculate how many samples to take from each date
unique_dates = df['date'].unique()
samples_per_date = n_samples // len(unique_dates)
remaining_samples = n_samples % len(unique_dates)

print(f"\nTaking {samples_per_date} samples from each of {len(unique_dates)} dates")
if remaining_samples > 0:
    print(f"Plus {remaining_samples} additional samples from first {remaining_samples} dates")

# Sample equal amounts from each date
rng, rng_ = jax.random.split(rng)
selected_indices = []

for i, date in enumerate(sorted(unique_dates)):
    date_samples = df[df['date'] == date]['sample_idx'].values
    
    # Determine how many samples to take from this date
    n_from_this_date = samples_per_date
    if i < remaining_samples:
        n_from_this_date += 1
    
    # Randomly sample from this date's samples
    if len(date_samples) >= n_from_this_date:
        rng, rng_date = jax.random.split(rng)
        chosen = jax.random.choice(
            rng_date,
            jnp.array(date_samples),
            shape=(n_from_this_date,),
            replace=False
        )
        selected_indices.extend(chosen.tolist())
    else:
        # If not enough samples for this date, take all available
        selected_indices.extend(date_samples.tolist())
        print(f"Warning: Date {date} only has {len(date_samples)} samples, needed {n_from_this_date}")

# Shuffle the selected indices and reshape into batches
rng, rng_shuffle = jax.random.split(rng)
selected_indices = jax.random.permutation(rng_shuffle, jnp.array(selected_indices))

# Ensure we have the right number of samples for batching
n_batches = len(selected_indices) // batch_size
selected_indices = selected_indices[:n_batches * batch_size]

sample_i = selected_indices.reshape(-1, batch_size).tolist()

# Verify the date distribution in our selection
selected_dates = [ds.get_date(idx) for batch in sample_i for idx in batch]
selected_df = pd.DataFrame({'date': selected_dates})

Date distribution in dataset:
date
2023-01-03    299
2023-01-04    489
2023-01-05    349
2023-01-06    382
2023-01-09    309
2023-01-10    436
2023-01-11    287
2023-01-12    468
2023-01-13    318
Name: count, dtype: int64

Taking 113 samples from each of 9 dates
Plus 7 additional samples from first 7 dates


In [12]:
# Analyze date distribution for each sample
date_distribution = {}
for batch_idx, batch in enumerate(sample_i):
    for sample_idx in batch:
        date = ds.get_date(sample_idx)
        if date not in date_distribution:
            date_distribution[date] = 0
        date_distribution[date] += 1

# Print date distribution summary
print("Date distribution across all samples:")
for date, count in sorted(date_distribution.items()):
    print(f"Date {date}: {count} samples")

print(f"\nTotal unique dates: {len(date_distribution)}")
print(f"Total samples: {sum(date_distribution.values())}")


Date distribution across all samples:
Date 2023-01-03: 114 samples
Date 2023-01-04: 114 samples
Date 2023-01-05: 114 samples
Date 2023-01-06: 114 samples
Date 2023-01-09: 114 samples
Date 2023-01-10: 114 samples
Date 2023-01-11: 114 samples
Date 2023-01-12: 113 samples
Date 2023-01-13: 113 samples

Total unique dates: 9
Total samples: 1024


In [13]:
# sample_i = pd.read_csv("/app/batches_genai_sample_indices_b64_bs16_ins100_cool20.json")


# sample_i = pd.read_json("/app/batches_genai_sample_indices_b64_bs16_ins100_cool20.json").values.tolist()


import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from collections import defaultdict

# Create a dictionary to track which dates appear in each sample (batch)
sample_date_composition = defaultdict(lambda: defaultdict(int))

for batch_idx, batch in enumerate(sample_i):
    for sample_idx in batch:
        date = ds.get_date(sample_idx)
        sample_date_composition[batch_idx][date] += 1

# Get all unique dates across all samples
all_dates = sorted(set(date for batch_dates in sample_date_composition.values() for date in batch_dates.keys()))

# Prepare data for stacked bar plot
n_samples = len(sample_i)
date_counts = np.zeros((len(all_dates), n_samples))

for batch_idx in range(n_samples):
    for date_idx, date in enumerate(all_dates):
        date_counts[date_idx, batch_idx] = sample_date_composition[batch_idx].get(date, 0)

# Create the stacked bar plot using Plotly
fig = go.Figure()

# Create color palette
colors = px.colors.qualitative.Set3
if len(all_dates) > len(colors):
    colors = colors * (len(all_dates) // len(colors) + 1)

# Add traces for each date
for date_idx, date in enumerate(all_dates):
    fig.add_trace(go.Bar(
        x=list(range(n_samples)),
        y=date_counts[date_idx],
        name=f'Date {date}',
        marker_color=colors[date_idx % len(colors)],
        opacity=0.8
    ))

# Update layout
fig.update_layout(
    title='Date Composition of Each Sample Batch',
    xaxis_title='Sample (Batch) Index',
    yaxis_title='Number of Indices from Each Date',
    barmode='stack',
    width=1200,
    height=600,
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    )
)

# Set x-axis ticks for better readability
if n_samples > 50:
    tick_positions = list(range(0, n_samples, max(1, n_samples // 20)))
    fig.update_xaxes(
        tickmode='array',
        tickvals=tick_positions,
        ticktext=[str(i) for i in tick_positions]
    )

fig.show()

# Print summary statistics
print(f"Total samples (batches): {n_samples}")
print(f"Batch size: {len(sample_i[0]) if sample_i else 0}")
print(f"Unique dates represented: {len(all_dates)}")
print(f"Date range: {min(all_dates)} to {max(all_dates)}")

Total samples (batches): 64
Batch size: 16
Unique dates represented: 9
Date range: 2023-01-03 to 2023-01-13


In [14]:
# Save sample_i to JSON file
import json

filename = "bathces_equal_sample_indices_b64_bs16_ins100_cool20.json"
with open(filename, 'w') as f:
    json.dump(sample_i, f)

print(f"Saved sample_i to {filename}")
print(f"Number of batches: {len(sample_i)}")
print(f"Batch size: {len(sample_i[0]) if sample_i else 0}")


Saved sample_i to bathces_equal_sample_indices_b64_bs16_ins100_cool20.json
Number of batches: 64
Batch size: 16


# ================= Scenario debugging =================

In [ ]:
# def construct_custom_msg(last_msg, current_book, sim):
#             ORDER_ID_i = 77777777
#             EVENT_TYPE = jnp.full((batch_size,), EVENT_TYPE_i)
#             SIDE = jnp.full((batch_size,), DIRECTION_i)
#             PRICE = last_msg[:, 3]  # Use previous price
#             DISPLAY_FLAG = jnp.ones((batch_size,), dtype=jnp.int32)

#             best_bid_info, best_ask_info = jax.vmap(sim.get_best_bid_and_ask_inclQuants)(current_book)

#             if DIRECTION_i == 0:
#                 best_volume = best_ask_info[:, 1]  # ← второй столбец = объём
#             else:
#                 best_volume = best_bid_info[:, 1]

#             # Decide volume based on config flag
#             if use_relative_volume:
#                 SIZE = (order_volume_ratio * best_volume).astype(jnp.int32)
#             else:
#                 SIZE = jnp.full((batch_size,), order_volume, dtype=jnp.int32)

#             print(f'SIZE: ', SIZE)

#             # Ensure SIZE is within allowed limits
#             SIZE = jnp.clip(SIZE, 1, 999999)

#             zeros = jnp.zeros((batch_size,), dtype=jnp.int32)
#             TIME_s = last_msg[:, 8]
#             TIME_ns = last_msg[:, 9]

#             msg = jnp.stack([
#                 jnp.full((batch_size,), ORDER_ID_i),
#                 EVENT_TYPE,
#                 SIDE,
#                 PRICE,
#                 DISPLAY_FLAG,
#                 SIZE,
#                 zeros, zeros,
#                 TIME_s, TIME_ns,
#                 zeros, zeros, zeros, zeros,
#             ], axis=1)

#             return msg.astype(jnp.int32)

# def run_historical_scenario(
#         n_samples: int,
#         batch_size: int,
#         ds: LOBSTER_Dataset,
#         rng: jax.dtypes.prng_key,
#         seq_len: int,
#         n_msgs: int,
#         n_gen_msgs: int,
#         train_state: TrainState,
#         model: nn.Module,
#         batchnorm: bool,
#         encoder: Dict[str, Tuple[jax.Array, jax.Array]],
#         stock_symbol: str,
#         n_vol_series: int = 500,
#         save_folder: str = './data_saved/',
#         tick_size: int = 100,
#         sample_top_n: int = -1,
#         sample_all: bool = False,
#         num_insertions: int = 2,
#         num_coolings: int = 2,
#         midprice_step_size=100,
#         EVENT_TYPE_i = 4,
#         DIRECTION_i = 0,
#         order_volume = 75,
#         use_sample_file: bool = False,
#         sample_file_path: Optional[str] = None,
#         start_batch: int = 0,
#         end_batch: int = -1
#     ):
#     """
#     Manual, step-by-step scenario runner: for each batch, processes messages one at a time,
#     updating the orderbook and tracking midprices, mimicking track_midprices_during_messages.
#     Saves processed messages, books, and midprices for each batch.

    
#     """
    
#     rng, rng_ = jax.random.split(rng)

#     if use_sample_file:
#         assert sample_file_path is not None, "Path to sample file not provided"
        
#         with open(sample_file_path, "r") as f:
#             sample_i_full = json.load(f)

#         sample_i = sample_i_full[start_batch:end_batch if end_batch != -1 else None]

#         for i, batch in enumerate(sample_i):
#             assert len(batch) == batch_size, f"Batch {i} has incorrect size {len(batch)}, expected {batch_size}"

#     else:
#         if sample_all:
#             sample_i = jnp.arange(
#                 len(ds) // batch_size * batch_size,
#                 dtype=jnp.int32
#             ).reshape(-1, batch_size).tolist()
#         else:
#             assert n_samples % batch_size == 0, 'n_samples must be divisible by batch_size'
#             sample_i = jax.random.choice(
#                 rng_,
#                 jnp.arange(len(ds), dtype=jnp.int32),
#                 shape=(n_samples // batch_size, batch_size),
#                 replace=False
#             ).tolist()

#     rng, rng_ = jax.random.split(rng)

    
#     save_folder = Path(save_folder)
#     save_folder.joinpath('msgs_decoded_doubled').mkdir(exist_ok=True, parents=True)
#     # save_folder.joinpath('l2_book_states_halved').mkdir(exist_ok=True, parents=True)
#     save_folder.joinpath('b_seq_gen_doubled').mkdir(exist_ok=True, parents=True)
#     save_folder.joinpath('mid_price').mkdir(exist_ok=True, parents=True)
#     base_save_folder = save_folder

#     for batch_i in tqdm(sample_i):
#         print('BATCH', batch_i)
#         m_seq, _, b_seq_pv, msg_seq_raw, book_l2_init = ds[batch_i]
#         m_seq = jnp.array(m_seq)
#         b_seq_pv = jnp.array(b_seq_pv)
#         msg_seq_raw = jnp.array(msg_seq_raw)
#         book_l2_init = jnp.array(book_l2_init)

#         current_book = book_l2_init

#         #=============#
#         # # Step 1: Prepare positions where to insert messages (accounting for prior insertions)
#         insertion_points = [n_msgs + (i + 1) * n_gen_msgs + i for i in range(num_insertions)]
#         insertion_points = [p for p in insertion_points if p <= msg_seq_raw.shape[1]]
#         print(f"[BATCH {batch_i}] Inserting custom messages at: {insertion_points}")

#         # # Step 2: Generate placeholder message using same logic as insert_custom_end (just 14-dim msg, no book logic yet)
#         # def construct_custom_msg(last_msg):
#         #     ORDER_ID_i = 77777777
#         #     EVENT_TYPE = jnp.full((batch_size,), EVENT_TYPE_i)
#         #     SIDE = jnp.full((batch_size,), DIRECTION_i)
#         #     PRICE = last_msg[:, 3]  # or use fixed jnp.full((batch_size,), 123456)
#         #     DISPLAY_FLAG = jnp.ones((batch_size,), dtype=jnp.int32)
            

#         #     #
#         #     SIZE = jnp.full((batch_size,), order_volume) # - need to get an actual best ask/bid volume and choose min(999999, available_best_volume)
#         #     #

#         #     zeros = jnp.zeros((batch_size,), dtype=jnp.int32)
#         #     TIME_s = last_msg[:, 8]
#         #     TIME_ns = last_msg[:, 9]

#         #     msg = jnp.stack([
#         #         jnp.full((batch_size,), ORDER_ID_i),
#         #         EVENT_TYPE,
#         #         SIDE,
#         #         PRICE,
#         #         DISPLAY_FLAG,
#         #         SIZE,
#         #         zeros, zeros,
#         #         TIME_s, TIME_ns,
#         #         zeros, zeros, zeros, zeros,
#         #     ], axis=1)

#         #     return msg.astype(jnp.int32)

        

#         # Step 3: Loop and insert messages
#         # for i, idx in enumerate(insertion_points):
#         #     custom_msg = construct_custom_msg(msg_seq_raw[:, idx - 1])
#         #     msg_seq_raw = jnp.concatenate([
#         #         msg_seq_raw[:, :idx, :],
#         #         custom_msg[:, None, :],  # shape (B, 1, 14)
#         #         msg_seq_raw[:, idx:, :]
#         #     ], axis=1)

#         for i, idx in enumerate(insertion_points):
#             # Simulate state before insertion
#             msg_prev = msg_seq_raw[:, idx - 1:idx, :]
#             sim_init, sim_state = inference.get_sims_vmap(current_book, msg_prev)

#             # Construct custom message using sim state and current book
#             custom_msg = construct_custom_msg(msg_prev[:, 0, :], sim_state, sim_init)

#             # Insert the message into sequence
#             msg_seq_raw = jnp.concatenate([
#                 msg_seq_raw[:, :idx, :],
#                 custom_msg[:, None, :],
#                 msg_seq_raw[:, idx:, :]
#             ], axis=1)

#         # print(f"[BATCH {batch_i}] msg_seq_raw shape after insertions: {msg_seq_raw.shape}")
#         #=============#

#         batch_size, T, msg_dim = msg_seq_raw.shape
#         current_book = book_l2_init

#         books = []
#         messages = []
#         midprices = []

#         for t in range(T):
#             msg = msg_seq_raw[:, t:t+1, :]
#             sim_init, sim_states = inference.get_sims_vmap(current_book, msg)
#             mid_price = inference.batched_get_safe_mid_price(sim_init, sim_states, tick_size)
#             full_l2_state = jax.vmap(sim_init.get_L2_state, in_axes=(0, None))(sim_states, current_book.shape[1])
#             current_book = full_l2_state[:, : current_book.shape[1]]
#             books.append(current_book)
#             messages.append(msg)
#             midprices.append(mid_price)

#         books = jnp.stack(books, axis=1)             # (batch, T, book_dim)
#         messages = jnp.concatenate(messages, axis=1) # (batch, T, msg_dim)
#         midprices = jnp.stack(midprices, axis=0)     # (T, batch)

#         # print(f"[BATCH {batch_i}] Finished all {T} steps")
#         # print(f"[BATCH {batch_i}] Final messages shape: {messages.shape}")
#         # print(f"[BATCH {batch_i}] Final books shape: {books.shape}")
#         # print(f"[BATCH {batch_i}] Final midprices shape: {midprices.shape}")

#         np.save(os.path.join(base_save_folder, 'msgs_decoded_doubled', f'msgs_decoded_doubled_batch_{batch_i}_iter_0.npy'), np.array(jax.device_get(messages)))
#         # np.save(os.path.join(base_save_folder, 'l2_book_states_halved', f'l2_book_states_halved_batch_{batch_i}_iter_0.npy'), np.array(jax.device_get(books)))
#         np.save(os.path.join(base_save_folder, 'mid_price', f'mid_price_batch_{batch_i}_iter_0.npy'), np.array(jax.device_get(midprices)))

#         # ========================
#         transform_L2_state_batch = jax.jit(jax.vmap(preproc.transform_L2_state, in_axes=(0, None, None)), static_argnums=(1, 2))

#         # Get midprices for each step: (T, B) → (B, T)
#         midprices_batched = midprices.T  # (B, T)
#         p_mid = midprices_batched[:, :, None]  # (B, T, 1)

#         # Add midprice as first column to each book state
#         books_with_mid = jnp.concatenate([p_mid, books], axis=-1)  # (B, T, 41)

#         # Transform each book+midprice into model input format
#         books_transformed = transform_L2_state_batch(books_with_mid, n_vol_series, tick_size)  # (B, T, D)

#         # Save transformed books
#         np.save(os.path.join(base_save_folder, 'b_seq_gen_doubled', f'b_seq_gen_doubled_batch_{batch_i}_iter_0.npy'), np.array(jax.device_get(books_transformed)))

In [ ]:
def construct_custom_msg(
    last_msg: jnp.ndarray,
    current_book: jnp.ndarray,
    sim_init,
    sim_state,
    DIRECTION_i: int,
    EVENT_TYPE_i: int,
    order_volume: int,
    use_relative_volume: bool,
    order_volume_ratio: float
) -> jnp.ndarray:
    """
    Construct a custom message (e.g., a large market order) based on the last message and current order book.
    """

    batch_size = last_msg.shape[0]
    ORDER_ID_i = 77777777

    EVENT_TYPE = jnp.full((batch_size,), EVENT_TYPE_i, dtype=jnp.int32)
    SIDE = jnp.full((batch_size,), DIRECTION_i, dtype=jnp.int32)
    PRICE = last_msg[:, 3]
    DISPLAY_FLAG = jnp.ones((batch_size,), dtype=jnp.int32)

    # Correctly call per-state bid/ask info
    best_bid_info, best_ask_info = jax.vmap(
        lambda state: sim_init.get_best_bid_and_ask_inclQuants(state)
    )(sim_state)

    # Get price and volume from current book state
    PRICE = jnp.where(
        DIRECTION_i == 0,
        best_ask_info[:, 0],  # best ask price
        best_bid_info[:, 0]   # best bid price
    )

    best_volume = jnp.where(
        DIRECTION_i == 0,
        best_ask_info[:, 1],  # volume at best ask
        best_bid_info[:, 1]   # volume at best bid
    )

    if use_relative_volume:
        SIZE = (order_volume_ratio * best_volume).astype(jnp.int32)
    else:
        SIZE = jnp.full((batch_size,), order_volume, dtype=jnp.int32)

    SIZE = jnp.clip(SIZE, 1, 999999)

    zeros = jnp.zeros((batch_size,), dtype=jnp.int32)
    TIME_s = last_msg[:, 8]
    TIME_ns = last_msg[:, 9]

    custom_msg = jnp.stack([
        jnp.full((batch_size,), ORDER_ID_i, dtype=jnp.int32),
        EVENT_TYPE,
        SIDE,
        PRICE,
        DISPLAY_FLAG,
        SIZE,
        zeros, zeros,
        TIME_s,
        TIME_ns,
        zeros, zeros, zeros, zeros
    ], axis=1)

    return custom_msg.astype(jnp.int32)

In [ ]:
def run_historical_scenario(
        n_samples: int,
        batch_size: int,
        ds: LOBSTER_Dataset,
        rng: jax.dtypes.prng_key,
        seq_len: int,
        n_msgs: int,
        n_gen_msgs: int,
        train_state: TrainState,
        model: nn.Module,
        batchnorm: bool,
        encoder: Dict[str, Tuple[jax.Array, jax.Array]],
        stock_symbol: str,
        n_vol_series: int = 500,
        save_folder: str = './data_saved/',
        tick_size: int = 100,
        sample_top_n: int = -1,
        sample_all: bool = False,
        num_insertions: int = 2,
        num_coolings: int = 2,
        midprice_step_size=100,
        EVENT_TYPE_i = 4,
        DIRECTION_i = 0,
        order_volume = 75,
        use_sample_file: bool = False,
        sample_file_path: Optional[str] = None,
        start_batch: int = 0,
        end_batch: int = -1
    ):


    rng, rng_ = jax.random.split(rng)

    if use_sample_file:
        assert sample_file_path is not None, "Path to sample file not provided"
        
        with open(sample_file_path, "r") as f:
            sample_i_full = json.load(f)

        sample_i = sample_i_full[start_batch:end_batch if end_batch != -1 else None]

        for i, batch in enumerate(sample_i):
            assert len(batch) == batch_size, f"Batch {i} has incorrect size {len(batch)}, expected {batch_size}"

    else:
        if sample_all:
            sample_i = jnp.arange(
                len(ds) // batch_size * batch_size,
                dtype=jnp.int32
            ).reshape(-1, batch_size).tolist()
        else:
            assert n_samples % batch_size == 0, 'n_samples must be divisible by batch_size'
            sample_i = jax.random.choice(
                rng_,
                jnp.arange(len(ds), dtype=jnp.int32),
                shape=(n_samples // batch_size, batch_size),
                replace=False
            ).tolist()

    rng, rng_ = jax.random.split(rng)
    
    save_folder = Path(save_folder)
    save_folder.joinpath('msgs_decoded_doubled').mkdir(exist_ok=True, parents=True)
    save_folder.joinpath('b_seq_gen_doubled').mkdir(exist_ok=True, parents=True)
    save_folder.joinpath('mid_price').mkdir(exist_ok=True, parents=True)
    
    for batch_i in tqdm(sample_i):
        print('BATCH', batch_i)
        m_seq, _, b_seq_pv, msg_seq_raw, book_l2_init = ds[batch_i]
        msg_seq_raw = jnp.array(msg_seq_raw)
        book_l2_init = jnp.array(book_l2_init)
        
        current_book = book_l2_init
        
        
        insertion_points = [n_msgs + (i + 1) * n_gen_msgs + i for i in range(num_insertions)]
        insertion_points = sorted([p for p in insertion_points if p <= msg_seq_raw.shape[1]])  
        print(f"[BATCH {batch_i}] Planned insertion points at indices: {insertion_points}")
        
        
        books = []
        messages = []
        midprices = []
        
        
        original_idx = 0
        steps_count = 0
        insertion_iter = 0

        
        
        
        while original_idx < msg_seq_raw.shape[1] or insertion_iter < len(insertion_points):
            if insertion_iter < len(insertion_points) and steps_count == insertion_points[insertion_iter]:
                if steps_count == 0:
                    last_msg = msg_seq_raw[:, 0, :]
                else:
                    last_msg = messages[-1]
                sim_init, sim_state = inference.get_sims_vmap(current_book, last_msg[:, None, :])
                custom_msg = construct_custom_msg(
                    last_msg,
                    current_book,
                    sim_init,
                    sim_state,
                    DIRECTION_i=DIRECTION_i,
                    EVENT_TYPE_i=EVENT_TYPE_i,
                    order_volume=order_volume,
                    use_relative_volume=use_relative_volume,
                    order_volume_ratio=order_volume_ratio
                )
                
                
                sim_init_ins, sim_state_ins = inference.get_sims_vmap(current_book, custom_msg[:, None, :])
                
                mid_price_ins = inference.batched_get_safe_mid_price(sim_init_ins, sim_state_ins, tick_size)
                
                full_l2_state_ins = jax.vmap(sim_init_ins.get_L2_state, in_axes=(0, None))(
                    sim_state_ins, current_book.shape[1]
                )
                current_book = full_l2_state_ins[:, : current_book.shape[1]]  # update current book to new state
                
                
                messages.append(custom_msg)
                books.append(current_book)
                midprices.append(mid_price_ins)
                
                steps_count += 1
                insertion_iter += 1
                continue
            
            if original_idx < msg_seq_raw.shape[1]:
                msg = msg_seq_raw[:, original_idx: original_idx + 1, :]
                sim_init_hist, sim_state_hist = inference.get_sims_vmap(current_book, msg)
                mid_price_hist = inference.batched_get_safe_mid_price(sim_init_hist, sim_state_hist, tick_size)
                full_l2_state_hist = jax.vmap(sim_init_hist.get_L2_state, in_axes=(0, None))(
                    sim_state_hist, current_book.shape[1]
                )
                current_book = full_l2_state_hist[:, : current_book.shape[1]]
                
                messages.append(msg[:, 0, :])
                books.append(current_book)
                midprices.append(mid_price_hist)
                
                original_idx += 1
                steps_count += 1
            else:
                print("No more historical messages. Awaiting remaining insertions...")
                break
        
        messages_arr = jnp.concatenate([m.reshape(m.shape[0], 1, m.shape[1]) for m in messages], axis=1)  # (batch, T_total, 14)
        books_arr = jnp.stack(books, axis=1)  # (batch, T_total, book_dim)
        midprices_arr = jnp.stack(midprices, axis=0)  # (T_total, batch)
        
        np.save(save_folder / 'msgs_decoded_doubled' / f'msgs_decoded_doubled_batch_{batch_i}_iter_0.npy',
                jax.device_get(messages_arr))
        np.save(save_folder / 'mid_price' / f'mid_price_batch_{batch_i}_iter_0.npy',
                jax.device_get(midprices_arr))
        
        transform_L2_state_batch = jax.jit(jax.vmap(preproc.transform_L2_state, in_axes=(0, None, None)), static_argnums=(1, 2))
        
        midprices_batched = midprices_arr.T[:, :, None]
        books_with_mid = jnp.concatenate([midprices_batched, books_arr], axis=-1)
        books_transformed = transform_L2_state_batch(books_with_mid, n_vol_series, tick_size)
        
        np.save(save_folder / 'b_seq_gen_doubled' / f'b_seq_gen_doubled_batch_{batch_i}_iter_0.npy',
                jax.device_get(books_transformed))

In [ ]:
results = run_historical_scenario(
        n_samples,
        batch_size,
        ds,
        rng,
        n_messages * Message_Tokenizer.MSG_LEN,
        n_messages,
        n_gen_msgs,
        state,
        model,
        args_ckpt.batchnorm,
        Vocab().ENCODING,
        stock,
        n_vol_series,
        exp_folder,
        tick_size,
        sample_top_n,
        sample_all,
        num_insertions,
        num_coolings,
        midprice_step_size,
        EVENT_TYPE_i,
        DIRECTION_i,
        order_volume,
        use_sample_file,
        sample_file_path,
        start_batch,
        end_batch
    )

# ================= Shifting =================

In [ ]:
# def run_historical_scenario(
#         n_samples: int,
#         batch_size: int,
#         ds: LOBSTER_Dataset,
#         rng: jax.dtypes.prng_key,
#         seq_len: int,
#         n_msgs: int,
#         n_gen_msgs: int,
#         train_state: TrainState,
#         model: nn.Module,
#         batchnorm: bool,
#         encoder: Dict[str, Tuple[jax.Array, jax.Array]],
#         stock_symbol: str,
#         n_vol_series: int = 500,
#         save_folder: str = './data_saved/',
#         tick_size: int = 100,
#         sample_top_n: int = -1,
#         sample_all: bool = False,
#         num_insertions: int = 2,
#         num_coolings: int = 2,
#         midprice_step_size=100,
#         EVENT_TYPE_i = 4,
#         DIRECTION_i = 0,
#         order_volume = 75,
#         use_sample_file: bool = False,
#         sample_file_path: Optional[str] = None,
#         start_batch: int = 0,
#         end_batch: int = -1
#     ):


#     rng, rng_ = jax.random.split(rng)

#     if use_sample_file:
#         assert sample_file_path is not None, "Path to sample file not provided"
        
#         with open(sample_file_path, "r") as f:
#             sample_i_full = json.load(f)

#         sample_i = sample_i_full[start_batch:end_batch if end_batch != -1 else None]

#         for i, batch in enumerate(sample_i):
#             assert len(batch) == batch_size, f"Batch {i} has incorrect size {len(batch)}, expected {batch_size}"

#     else:
#         if sample_all:
#             sample_i = jnp.arange(
#                 len(ds) // batch_size * batch_size,
#                 dtype=jnp.int32
#             ).reshape(-1, batch_size).tolist()
#         else:
#             assert n_samples % batch_size == 0, 'n_samples must be divisible by batch_size'
#             sample_i = jax.random.choice(
#                 rng_,
#                 jnp.arange(len(ds), dtype=jnp.int32),
#                 shape=(n_samples // batch_size, batch_size),
#                 replace=False
#             ).tolist()

#     rng, rng_ = jax.random.split(rng)
    
#     # ... (parameters and initial setup unchanged) ...
#     save_folder = Path(save_folder)
#     # Prepare directories for saving outputs (if not already existing)
#     save_folder.joinpath('msgs_decoded_doubled').mkdir(exist_ok=True, parents=True)
#     save_folder.joinpath('b_seq_gen_doubled').mkdir(exist_ok=True, parents=True)
#     save_folder.joinpath('mid_price').mkdir(exist_ok=True, parents=True)
    
#     for batch_i in tqdm(sample_i):
#         print('BATCH', batch_i)
#         m_seq, _, b_seq_pv, msg_seq_raw, book_l2_init = ds[batch_i]
#         # Convert to JAX arrays for processing
#         msg_seq_raw = jnp.array(msg_seq_raw)
#         book_l2_init = jnp.array(book_l2_init)
        
#         current_book = book_l2_init  # initialize order book state for this batch
        
#         # Determine positions (indices) at which to insert custom messages.
#         insertion_points = [n_msgs + (i + 1) * n_gen_msgs + i for i in range(num_insertions)]
#         insertion_points = sorted([p for p in insertion_points if p <= msg_seq_raw.shape[1]])  
#         # (Filtered to ensure insertion index is within sequence length; adjust if using generation beyond historical length)
#         print(f"[BATCH {batch_i}] Planned insertion points at indices: {insertion_points}")
        
#         # Prepare lists to collect the results
#         books = []
#         messages = []
#         midprices = []
        
#         # Pointers and counters for looping
#         original_idx = 0             # index in the original message sequence
#         steps_count = 0              # total messages processed (including insertions)
#         insertion_iter = 0           # how many insertions have been applied so far

        
        
#         # Loop through the sequence, including custom insertions
#         # ... (внутри цикла while, перед применением каждого исторического сообщения)
#         # if original_idx < msg_seq_raw.shape[1]:

        
#         PRICE_i = 3
#         SIZE_i = 5
#         DISPLAY_FLAG_i = 6

#         insertion_pointer = 0  # текущий указатель в списке insertion_points

#         while original_idx < msg_seq_raw.shape[1]:
#             # 1. ВСТАВКА КАСТОМНОГО ОРДЕРА, если требуется
#             # Вставка кастомного сообщения в нужный момент
#             if insertion_iter < len(insertion_points) and steps_count == insertion_points[insertion_iter]:
#                 n_levels = current_book.shape[1] // 4
#                 best_bid_price = current_book[:, 0]
#                 best_ask_price = current_book[:, n_levels]
#                 best_ask_volume = current_book[:, n_levels + 1]

#                 direction = 1  # агрессивный BID (ударяем по ASK)
#                 price = best_ask_price
#                 volume = best_ask_volume  # <-- теперь берём реальный доступный объём

#                 custom_msg = jnp.zeros_like(msg_seq_raw[:, 0:1, :])
#                 custom_msg = custom_msg.at[:, 0, EVENT_TYPE_i].set(1)  # Limit
#                 custom_msg = custom_msg.at[:, 0, DIRECTION_i].set(direction)
#                 custom_msg = custom_msg.at[:, 0, PRICE_i].set(price)
#                 custom_msg = custom_msg.at[:, 0, SIZE_i].set(volume)
#                 custom_msg = custom_msg.at[:, 0, DISPLAY_FLAG_i].set(999)  # special marker

#                 print(f"[INSERT {insertion_iter}] @ step {steps_count}")
#                 print(f"→ Insert custom LIMIT {'BID' if direction==0 else 'ASK'} @ {price[0]} | volume: {volume}")
#                 print(f"→ Book best_bid: {best_bid_price[0]}, best_ask: {best_ask_price[0]}")

#                 sim_init_custom, sim_state_custom = inference.get_sims_vmap(current_book, custom_msg)
#                 mid_price_custom = inference.batched_get_safe_mid_price(sim_init_custom, sim_state_custom, tick_size)
#                 print(f"→ Midprice after insert: {mid_price_custom[0]}")

#                 full_l2_custom = jax.vmap(sim_init_custom.get_L2_state, in_axes=(0, None))(
#                     sim_state_custom, current_book.shape[1]
#                 )
#                 current_book = full_l2_custom[:, : current_book.shape[1]]

#                 messages.append(custom_msg[:, 0, :])
#                 books.append(current_book)
#                 midprices.append(mid_price_custom)

#                 insertion_iter += 1
#                 steps_count += 1
#                 continue

#             # 2. ОБРАБОТКА ИСТОРИЧЕСКОГО СООБЩЕНИЯ
#             msg = msg_seq_raw[:, original_idx: original_idx + 1, :]

#             event_type = msg[0, 0, EVENT_TYPE_i]
#             direction = msg[0, 0, DIRECTION_i]

#             if event_type == 1:
#                 n_levels = current_book.shape[1] // 4
#                 best_bid = current_book[:, 0]
#                 best_ask = current_book[:, n_levels]
#                 original_price = msg[:, 0, PRICE_i]

#                 shifted = False
#                 if direction == 0:  # bid
#                     needs_shift = original_price >= best_ask
#                     if needs_shift.any():
#                         shifted = True
#                         new_price = jnp.where(needs_shift, best_ask - tick_size, original_price)
#                         msg = msg.at[:, 0, PRICE_i].set(new_price)
#                 else:  # ask
#                     needs_shift = original_price <= best_bid
#                     if needs_shift.any():
#                         shifted = True
#                         new_price = jnp.where(needs_shift, best_bid + tick_size, original_price)
#                         msg = msg.at[:, 0, PRICE_i].set(new_price)

#                 if shifted:
#                     print(f"[HIST] shift @ step {steps_count} | {'bid' if direction==0 else 'ask'}")
#                     print(f"→ Old price: {original_price[0]}, New price: {msg[:,0,PRICE_i][0]}")
#                     print(f"→ Book best_bid: {best_bid[0]}, best_ask: {best_ask[0]}")

#             # применяем историческое сообщение
#             sim_init_hist, sim_state_hist = inference.get_sims_vmap(current_book, msg)
#             mid_price_hist = inference.batched_get_safe_mid_price(sim_init_hist, sim_state_hist, tick_size)
#             full_l2_state_hist = jax.vmap(sim_init_hist.get_L2_state, in_axes=(0, None))(
#                 sim_state_hist, current_book.shape[1]
#             )
#             current_book = full_l2_state_hist[:, : current_book.shape[1]]

#             messages.append(msg[:, 0, :])
#             books.append(current_book)
#             midprices.append(mid_price_hist)

#             original_idx += 1
#             steps_count += 1
        
#         # Convert collected lists to JAX arrays for saving
#         messages_arr = jnp.concatenate([m.reshape(m.shape[0], 1, m.shape[1]) for m in messages], axis=1)  # (batch, T_total, 14)
#         books_arr = jnp.stack(books, axis=1)  # (batch, T_total, book_dim)
#         midprices_arr = jnp.stack(midprices, axis=0)  # (T_total, batch)
        
#         # Save the results to .npy files
#         np.save(save_folder / 'msgs_decoded_doubled' / f'msgs_decoded_doubled_batch_{batch_i}_iter_0.npy',
#                 jax.device_get(messages_arr))
#         np.save(save_folder / 'mid_price' / f'mid_price_batch_{batch_i}_iter_0.npy',
#                 jax.device_get(midprices_arr))
        
#         transform_L2_state_batch = jax.jit(jax.vmap(preproc.transform_L2_state, in_axes=(0, None, None)), static_argnums=(1, 2))
        
#         # Transform the order book states (with midprice) for model input if needed
#         midprices_batched = midprices_arr.T[:, :, None]  # shape (batch, T_total, 1)
#         books_with_mid = jnp.concatenate([midprices_batched, books_arr], axis=-1)  # (batch, T_total, 1+book_dim)
#         books_transformed = transform_L2_state_batch(books_with_mid, n_vol_series, tick_size)
        
#         np.save(save_folder / 'b_seq_gen_doubled' / f'b_seq_gen_doubled_batch_{batch_i}_iter_0.npy',
#                 jax.device_get(books_transformed))
        



# # shifting only when our inserion order!!

In [ ]:
# results = run_historical_scenario(
#         n_samples,
#         batch_size,
#         ds,
#         rng,
#         n_messages * Message_Tokenizer.MSG_LEN,
#         n_messages,
#         n_gen_msgs,
#         state,
#         model,
#         args_ckpt.batchnorm,
#         Vocab().ENCODING,
#         stock,
#         n_vol_series,
#         exp_folder,
#         tick_size,
#         sample_top_n,
#         sample_all,
#         num_insertions,
#         num_coolings,
#         midprice_step_size,
#         EVENT_TYPE_i,
#         DIRECTION_i,
#         order_volume,
#         use_sample_file,
#         sample_file_path,
#         start_batch,
#         end_batch
#     )